<a href="https://www.kaggle.com/code/insanejsk/bm25-baseline-on-ms-marco?scriptVersionId=326335783" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [3]:
!pip install BM25 rank_bm25 --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 1.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 752.4/752.4 kB 5.9 MB/s eta 0:00:0000:0100:01


In [2]:
from datasets import load_dataset
import json
from pathlib import Path

dataset = load_dataset(
    "ms_marco",
    "v2.1",
    split="train",
    streaming=True
)

sample = next(iter(dataset))
print("Keys:", sample.keys())
print("Query:", sample["query"])
print("Passages:", sample["passages"])

README.md: 0.00B [00:00, ?B/s]

Keys: dict_keys(['answers', 'passages', 'query', 'query_id', 'query_type', 'wellFormedAnswers'])
Query: )what was the immediate impact of the success of the manhattan project?
Passages: {'is_selected': [1, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'passage_text': ['The presence of communication amid scientific minds was equally important to the success of the Manhattan Project as scientific intellect was. The only cloud hanging over the impressive achievement of the atomic researchers and engineers is what their success truly meant; hundreds of thousands of innocent lives obliterated.', 'The Manhattan Project and its atomic bomb helped bring an end to World War II. Its legacy of peaceful uses of atomic energy continues to have an impact on history and science.', 'Essay on The Manhattan Project - The Manhattan Project The Manhattan Project was to see if making an atomic bomb possible. The success of this project would forever change the world forever making it known that something this powerful can b

In [3]:
import random
from pprint import pprint

def extract_triples(dataset, max_triples=50_000):
    """
    Extract (query, positive, negative) triples from MS MARCO.
    - positive  = the passage where is_selected == 1
    - negative  = one random passage where is_selected == 0
                  ONE negative per query for now.
    """
    triples = []
    skipped = 0

    for sample in dataset:
        passages   = sample["passages"]
        is_selected = passages["is_selected"]
        texts       = passages["passage_text"]

        # ── Find positive ────────────────────────────────────
        pos_indices = [i for i, s in enumerate(is_selected) if s == 1]
        neg_indices = [i for i, s in enumerate(is_selected) if s == 0]

        # Skip if no positive exists
        if not pos_indices or not neg_indices:
            skipped += 1
            continue

        positive = texts[pos_indices[0]]           # take first positive
        negative = texts[random.choice(neg_indices)]  # random negative for now

        triples.append({
            "query":    sample["query"],
            "positive": positive,
            "negative": negative
        })

        if len(triples) >= max_triples:
            break

    print(f"Built {len(triples)} triples, skipped {skipped} samples")
    return triples

#Training Set
dataset = load_dataset(
    "ms_marco",
    "v2.1",
    split="train",
    streaming=True
)

triples = extract_triples(dataset, max_triples=200000)

print("\n--- Sample triple ---")
pprint(triples[0])
output_path = "triples_train.jsonl"

with open(output_path, "w") as f:
    for triple in triples:
        f.write(json.dumps(triple) + "\n")

print(f"\nSaved to {output_path}")

Built 200000 triples, skipped 126756 samples

--- Sample triple ---
{'negative': 'The Manhattan Project. This once classified photograph features '
             'the first atomic bomb — a weapon that atomic scientists had '
             'nicknamed Gadget.. The nuclear age began on July 16, 1945, when '
             'it was detonated in the New Mexico desert.',
 'positive': 'The presence of communication amid scientific minds was equally '
             'important to the success of the Manhattan Project as scientific '
             'intellect was. The only cloud hanging over the impressive '
             'achievement of the atomic researchers and engineers is what '
             'their success truly meant; hundreds of thousands of innocent '
             'lives obliterated.',
 'query': ')what was the immediate impact of the success of the manhattan '
          'project?'}

Saved to triples_train.jsonl


In [4]:
triples[0]

{'query': ')what was the immediate impact of the success of the manhattan project?',
 'positive': 'The presence of communication amid scientific minds was equally important to the success of the Manhattan Project as scientific intellect was. The only cloud hanging over the impressive achievement of the atomic researchers and engineers is what their success truly meant; hundreds of thousands of innocent lives obliterated.',
 'negative': 'The Manhattan Project. This once classified photograph features the first atomic bomb — a weapon that atomic scientists had nicknamed Gadget.. The nuclear age began on July 16, 1945, when it was detonated in the New Mexico desert.'}

In [5]:
#Validation Set
dataset_val = load_dataset(
    "ms_marco",
    "v2.1",
    split="validation",
    streaming=True
)

triples_val = extract_triples(dataset_val, max_triples=5_000)

# Save
with open("triples_val.jsonl", "w") as f:
    for triple in triples_val:
        f.write(json.dumps(triple) + "\n")

print(f"Validation set: {len(triples_val)} triples")

Built 5000 triples, skipped 2844 samples
Validation set: 5000 triples


In [6]:
triples_val[0]

{'query': '. what is a corporation?',
 'positive': "McDonald's Corporation is one of the most recognizable corporations in the world. A corporation is a company or group of people authorized to act as a single entity (legally a person) and recognized as such in law. Early incorporated entities were established by charter (i.e. by an ad hoc act granted by a monarch or passed by a parliament or legislature).",
 'negative': "Corporations are owned by their stockholders (shareholders) who share in profits and losses generated through the firm's operations, and have three distinct characteristics (1) Legal existence: a firm can (like a person) buy, sell, own, enter into a contract, and sue other persons and firms, and be sued by them."}

### Baseline - BM25

$$
\text{score}(query, passage) =
\sum_{\text{term}}
\frac{\text{IDF}(\text{term}) \times \text{TF}(\text{term}, passage) \times (k_1 + 1)}
{\text{TF} + k_1 \times \left(1 - b + b \times \frac{|doc|}{avgdl}\right)}
$$

Where, for each query, we retrieve from a pool of:
   - its positive passage
   - all other negatives in the val set (sampled)
This is called in-pool evaluation

In [7]:
from rank_bm25 import BM25Okapi
import json
import numpy as np
from tqdm import tqdm

#Baseline BM25 on val set

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

val_triples = load_jsonl("triples_val.jsonl")
print(f"Loaded {len(val_triples)} validation triples")

# retrieval corpus from val triples 
corpus = []
passage_to_idx = {}

for triple in val_triples:
    for key in ["positive", "negative"]:
        p = triple[key]
        if p not in passage_to_idx:
            passage_to_idx[p] = len(corpus)
            corpus.append(p)

print(f"Corpus size: {len(corpus)} unique passages")

# BM25 works on token lists, not strings
tokenized_corpus = [doc.lower().split() for doc in corpus]
bm25 = BM25Okapi(tokenized_corpus)
print("BM25 index built")

# Recall@10
def recall_at_k(relevant_idx, retrieved_indices, k=10):
    """Did the positive appear in top-K results?"""
    return int(relevant_idx in retrieved_indices[:k])

# MRR (Mean Reciprocal Rank)
def reciprocal_rank(relevant_idx, retrieved_indices):
    """1/rank of the first relevant result. 0 if not found."""
    if relevant_idx in retrieved_indices:
        rank = retrieved_indices.index(relevant_idx) + 1  # 1-indexed
        return 1.0 / rank
    return 0.0

recalls = []
mrr_scores = []

for triple in tqdm(val_triples, desc="Evaluating BM25"):
    query     = triple["query"].lower().split()
    pos_text  = triple["positive"]
    pos_idx   = passage_to_idx[pos_text]

    # BM25 scores for this query over entire corpus
    scores = bm25.get_scores(query)

    # Get indices sorted by score descending
    ranked_indices = np.argsort(scores)[::-1].tolist()

    recalls.append(recall_at_k(pos_idx, ranked_indices, k=10))
    mrr_scores.append(reciprocal_rank(pos_idx, ranked_indices))

print("\n=== BM25 Baseline Results ===")
print(f"Recall@10 : {np.mean(recalls):.4f}")
print(f"MRR       : {np.mean(mrr_scores):.4f}")

Loaded 5000 validation triples
Corpus size: 9945 unique passages
BM25 index built


Evaluating BM25: 100%|██████████| 5000/5000 [01:48<00:00, 45.99it/s]


=== BM25 Baseline Results ===
Recall@10 : 0.7680
MRR       : 0.5805


In [ ]:
# ============================================================
# Parallelized BM25 hard negative mining
# ============================================================

from rank_bm25 import BM25Okapi
import json
import numpy as np
from tqdm import tqdm
from multiprocessing import Pool, cpu_count
import os

# ── Load data ────────────────────────────────────────────────
with open("/kaggle/input/datasets/insanejsk/nse-dr/triples_train.jsonl") as f:
    triples = [json.loads(line) for line in f]

corpus = list({t["positive"] for t in triples} |
              {t["negative"] for t in triples})
corpus_idx = {p: i for i, p in enumerate(corpus)}

print(f"Triples : {len(triples)}")
print(f"Corpus  : {len(corpus)} passages")
print(f"CPUs    : {cpu_count()}")

# ── Tokenize corpus once ─────────────────────────────────────
print("Tokenizing corpus...")
tokenized_corpus = [doc.lower().split() for doc in corpus]

# ── Worker function ──────────────────────────────────────────
# Must be top-level for multiprocessing to pickle it
def process_chunk(args):
    chunk, tokenized_corpus, corpus, corpus_idx = args
    bm25 = BM25Okapi(tokenized_corpus)  # each worker builds its own index
    results = []

    for triple in chunk:
        query    = triple["query"]
        positive = triple["positive"]
        pos_idx  = corpus_idx.get(positive, -1)

        scores  = bm25.get_scores(query.lower().split())
        ranked  = np.argsort(scores)[::-1].tolist()

        hard_candidates = [corpus[i] for i in ranked
                           if i != pos_idx][:10]

        if not hard_candidates:
            continue

        results.append({
            "query"   : query,
            "positive": positive,
            "negative": hard_candidates[0],
        })

    return results

# ── Split into chunks ────────────────────────────────────────
n_workers  = cpu_count()
chunk_size = len(triples) // n_workers
chunks     = [triples[i:i+chunk_size]
              for i in range(0, len(triples), chunk_size)]

print(f"Splitting into {len(chunks)} chunks of ~{chunk_size} each")
print("Starting parallel mining...")

# ── Run parallel ─────────────────────────────────────────────
args = [(chunk, tokenized_corpus, corpus, corpus_idx)
        for chunk in chunks]

with Pool(processes=n_workers) as pool:
    results = list(tqdm(
        pool.imap(process_chunk, args),
        total=len(chunks),
        desc="Chunks completed"
    ))

# ── Flatten results ──────────────────────────────────────────
hard_triples = [t for chunk_result in results for t in chunk_result]
print(f"\nGenerated {len(hard_triples)} hard negative triples")

# ── Save ─────────────────────────────────────────────────────
with open("triples_train_200k_hard.jsonl", "w") as f:
    for t in hard_triples:
        f.write(json.dumps(t) + "\n")

print("Saved to triples_train_200k_hard.jsonl")

# ── Sanity check ─────────────────────────────────────────────
ex = hard_triples[0]
print(f"\n--- Example ---")
print(f"Query    : {ex['query']}")
print(f"Positive : {ex['positive'][:100]}...")
print(f"Hard neg : {ex['negative'][:100]}...")

Triples : 200000
Corpus  : 393154 passages
CPUs    : 4
Tokenizing corpus...
Splitting into 4 chunks of ~50000 each
Starting parallel mining...


Chunks completed:   0%|          | 0/4 [00:00<?, ?it/s]

In [1]:
from multiprocessing import cpu_count
print(cpu_count())

4
